# Brick 12 - ablations, E1 features, E2 calibration

**Gate:** Ablations, E1 variants and the E2 conformal coverage/abstention numbers.

The narrative report for this brick is `reports/brick12_enhancements.md`. The cells below load the saved artifacts; they do not retrain anything.

In [1]:
import json
from pathlib import Path
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
FIGURES = ROOT / "figures"

def md(name):
    p = REPORTS / name
    display(Markdown(p.read_text(encoding="utf-8"))) if p.exists() else print("missing", name)

def js(name, keys=None, indent=2):
    p = REPORTS / name
    if not p.exists():
        print("missing", name); return None
    d = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps({k: d[k] for k in keys} if keys else d, indent=indent)[:6000])
    return d


In [2]:
md('brick12_enhancements.md')
js('brick12_enhancements.json', ['e2_calibration_conformal'])

# Brick 12 - ablations and section 15 enhancements

Mode S, Split G, XGBoost with the tuned Brick 8 parameters. Training rows 40,000 (capped), test rows 61,592.

## Ablations and feature sets

| config               |   macro_f1 |   accuracy |   n_features |
|:---------------------|-----------:|-----------:|-------------:|
| 18_moe               |   0.450142 |   0.452218 |           18 |
| drug_only            |   0.24544  |   0.346165 |           18 |
| food_only            |   0.529636 |   0.54434  |           18 |
| shuffled_food        |   0.376107 |   0.376461 |           18 |
| all_3780_features    |   0.494601 |   0.504867 |           18 |
| 18+E1a_drug_metadata |   0.444866 |   0.446714 |           19 |
| 18+E1b_food_class    |   0.442084 |   0.4441   |           28 |
| 18+E1c_fingerprints  |   0.448223 |   0.475662 |          530 |
| 18+E1abc_all         |   0.448551 |   0.473519 |          541 |

## E2 calibration and split conformal

Variant: 18_moe; q_hat = 0.7251 at a 90% target.
Test coverage 0.887, average prediction-set size 2.46, abstention rate 0.895.
Argmax accuracy 0.4230; selective accuracy 0.5850 on the 6,461 non-abstained pairs.
Drug-level bootstrap 95% CI for macro-F1: [0.35498440994610814, 0.46486675173257297].

Reading: drug- and food-only models show whether the pair interaction adds signal. A large drop under shuffled food pairing is the direct evidence that the model uses the pairing.

{
  "e2_calibration_conformal": {
    "variant": "18_moe",
    "q_hat": 0.7250685153777204,
    "target_coverage": 0.9,
    "coverage": 0.8866573580984544,
    "avg_set_size": 2.4636478763475775,
    "abstain_rate": 0.8951000129886998,
    "accuracy_all": 0.42302571762566565,
    "selective_accuracy": 0.5850487540628385,
    "selective_macro_f1": 0.4455002538390418,
    "seconds": 3.6,
    "bootstrap_drug_level": {
      "replicates": 1000,
      "n_drugs": 55,
      "macro_f1_ci95": [
        0.35498440994610814,
        0.46486675173257297
      ],
      "accuracy_ci95": [
        0.3628337543963215,
        0.48357051831977
      ]
    }
  }
}


{'mode': 'S',
 'split': 'G',
 'train_rows': 40000,
 'test_rows': 61592,
 'features': {'18_moe': {'accuracy': 0.4522178204961683,
   'macro_f1': 0.45014237972466775,
   'per_class': {'0': {'precision': 0.3740492393915132,
     'recall': 0.43118366405168435,
     'f1': 0.40058949624866025,
     'support': 17336},
    '1': {'precision': 0.5710159995379195,
     'recall': 0.4155702215309597,
     'f1': 0.48104715099021944,
     'support': 23789},
    '2': {'precision': 0.431858407079646,
     'recall': 0.512630087457859,
     'f1': 0.46879049193512357,
     'support': 20467}},
   'confusion_matrix': [[7475, 4051, 5810],
    [5910, 9886, 7993],
    [6599, 3376, 10492]],
   'seconds': 2.7}},
 'ablations': {'drug_only': {'accuracy': 0.3461650863748539,
   'macro_f1': 0.2454404647536165,
   'per_class': {'0': {'precision': 0.3895790200138026,
     'recall': 0.06512459621596678,
     'f1': 0.11159434615004447,
     'support': 17336},
    '1': {'precision': 0.80758137618459,
     'recall': 0.082

In [3]:
# Reproduce this brick from scratch (long running):
# !cd .. && PYTHONPATH=src python src/brick12_*.py